In [ ]:
import pandas as pd
from scipy.interpolate import interp1d
import numpy as np
import pyam

# Clean USGS MCS data

In [ ]:
df_usgs_mcs_raw = pd.read_csv(r'data/data_minerals/USGS/Mineral_commodity_summary_2025/MCS2025_World_Data.csv')

In [ ]:
def restructure_mcs(df):
    # Step 1: Keep only rows where COUNTRY contains 'World'
    df_world = df[df['COUNTRY'].str.contains("World", case=False, na=False)].copy()
    df_world['COUNTRY'] = 'World total'

    # Step 2: Standardize column names for easier parsing
    df_world.rename(columns={
        'PROD_2023': '2023_PROD',
        'PROD_EST_ 2024': '2024_PROD',
        'PROD_NOTES': 'PROD_NOTES',
        'CAP_2023': '2023_CAP',
        'CAP_EST_ 2024': '2024_CAP',
        'CAP_NOTES': 'CAP_NOTES',
        'RESERVES_2024': '2024_RES',
        'RESERVE_NOTES': 'RES_NOTES'
    }, inplace=True)

    # Step 3: Melt the production, capacity, and reserves values
    value_df = pd.melt(
        df_world,
        id_vars=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS'],
        value_vars=['2023_PROD', '2024_PROD', '2023_CAP', '2024_CAP', '2024_RES'],
        var_name='YEAR_DATATYPE',
        value_name='VALUE'
    )

    # Step 4: Extract YEAR and DATA_TYPE
    value_df[['YEAR', 'DATA_TYPE']] = value_df['YEAR_DATATYPE'].str.extract(r'(\d{4})_(PROD|CAP|RES)')
    value_df['DATA_TYPE'] = value_df['DATA_TYPE'].map({'PROD': 'Production', 'CAP': 'Capacity', 'RES': 'Reserves'})

    # Step 5: Melt the notes in long format and map their types
    note_df = pd.melt(
        df_world,
        id_vars=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS'],
        value_vars=['PROD_NOTES', 'CAP_NOTES', 'RES_NOTES'],
        var_name='NOTE_COLUMN',
        value_name='NOTES'
    )
    note_df['DATA_TYPE'] = note_df['NOTE_COLUMN'].str.extract(r'(PROD|CAP|RES)')[0].map({'PROD': 'Production', 'CAP': 'Capacity', 'RES': 'Reserves'})
    note_df.drop(columns='NOTE_COLUMN', inplace=True)

    # Step 6: Merge values with corresponding notes
    merged_df = pd.merge(
        value_df,
        note_df,
        on=['COMMODITY', 'COUNTRY', 'TYPE', 'UNIT_MEAS', 'DATA_TYPE'],
        how='left'
    )

    # Step 7: Drop rows where VALUE is NaN and sort
    cleaned_df = merged_df.dropna(subset=['VALUE']).sort_values(by='COMMODITY').reset_index(drop=True)

    # Step 8: Return only relevant columns
    return cleaned_df[['COMMODITY', 'COUNTRY', 'TYPE', 'DATA_TYPE', 'UNIT_MEAS', 'YEAR', 'VALUE', 'NOTES']]

In [ ]:
# Apply the function
df_usgs_mcs = restructure_mcs(df_usgs_mcs_raw)

In [ ]:
df_usgs_mcs

In [ ]:
df_usgs_mcs['COMMODITY'] = df_usgs_mcs['COMMODITY'].replace({
    "Zirconium and Hafnium": "Zirconium"})

In [ ]:
df_usgs_mcs

In [ ]:
df_usgs_mcs.to_csv(r'data/data_minerals/df_usgs_mcs.csv', index=False)

# Get SSP data from IIASA database

In [ ]:
pyam.iiasa.platforms()

In [ ]:
conn = pyam.iiasa.Connection()
conn.valid_connections

In [ ]:
conn_ssp = pyam.iiasa.Connection('ssp')

In [ ]:
conn_ssp.models()

In [ ]:
conn_ssp.scenarios()

In [ ]:
conn_ssp.regions()

In [ ]:
df_ssp = pyam.read_iiasa(
    "ssp",
    variable=["GDP|PPP", "GDP|PPP [per capita]", "Population", "Population|Urban|Share", "Population|Urban [Share]"],
    region="World",
    meta=True,
)

In [ ]:
df_ssp.timeseries().reset_index().to_csv(r'data/data_ssp/iamdf_ssp.csv', index=False)

# IEA scenario data

In [ ]:
df_iea = pd.read_csv(r'data/data_iea/WEO2024_AnnexA_Free_Dataset_World.csv')

In [ ]:
# For CO2 calculations

# We keep only CO2 columns
filtered_categories = [
    "CO2 combustion",
]

filtered_product = [
    "Total"
]

filtered_flows = [
    "Total energy supply"
]

filtered_scenarios = [
    "Stated Policies Scenario",
    "Net Zero Emissions by 2050 Scenario"
]

filtered_year = [
    2022, 
    2023,
    2030,
    2035,
    2040,
    2050
]

df_iea_co2 = df_iea[df_iea["CATEGORY"].isin(filtered_categories)]
df_iea_co2 = df_iea_co2[df_iea_co2["PRODUCT"].isin(filtered_product)]
df_iea_co2 = df_iea_co2[df_iea_co2["FLOW"].isin(filtered_flows)]
df_iea_co2 = df_iea_co2[df_iea_co2["SCENARIO"].isin(filtered_scenarios)]
df_iea_co2 = df_iea_co2[df_iea_co2["YEAR"].isin(filtered_year)]
df_iea_co2

In [ ]:
df_iea_co2.to_csv(r'data/data_iea/df_iea_co2.csv', index=False)